# Single-cell perturbations: split and baseline workflow

This notebook reuses a fixed pseudo-target split and evaluates a first PCA + Ridge baseline.

**Split:** hold out all but 17 `T cells CD4+` cell–drug pairs; keep other cell types in training; seed 42. All baselines reuse the saved pair IDs.

The JSON stores pair identities because it records **which rows are training versus validation**. Gene-level DE values remain in `de_train.parquet`; the notebook selects them into `y_train` and `y_val`.

This baseline uses cell/drug identities as input, compresses training gene-response targets with PCA, predicts PC scores with Ridge, then reconstructs gene-level predictions. It is a simple baseline, not a molecular-structure model.

In [3]:
%pip install kagglehub pandas pyarrow
%run download.py

from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = Path(path)

Note: you may need to restart the kernel to use updated packages.
Resuming download from 751828992 bytes (3359309787 bytes left)...
Resuming download to C:\Users\73714\.cache\kagglehub\competitions\open-problems-single-cell-perturbations.archive (751828992/4111138779) bytes left.


100%|██████████| 3.83G/3.83G [01:03<00:00, 52.8MB/s]

Extracting files...


Path to competition files: C:\Users\73714\.cache\kagglehub\competitions\open-problems-single-cell-perturbations


## 1. Load labeled DE data

Each row contains one cell-type–drug pair and its gene-level DE target values.

In [ ]:
import pandas as pd
from IPython.display import display
from split_data import load_or_create_split, apply_split, get_gene_columns
from baseline_pca_ridge import fit_predict, mrrmse
from baseline_ridge import fit_predict as fit_predict_ridge

VALIDATION_FRACTION = 0.2
RANDOM_SEED = 42
de_train = pd.read_parquet(DATA_DIR / "de_train.parquet")
id_map = pd.read_csv(DATA_DIR / "id_map.csv")
gene_cols = get_gene_columns(de_train)

print(f"Labeled pairs: {len(de_train)}")
print(f"DE target genes: {len(gene_cols)}")
print(f"Official test pairs: {len(id_map)}")
display(de_train.groupby("cell_type").size().rename("available_pairs").to_frame())

Labeled pairs: 614
DE target genes: 18211
Official test pairs: 255


,available_pairs
cell_type,
B cells,17
Myeloid cells,17
NK cells,146
T cells CD4+,146
T cells CD8+,142
T regulatory cells,146


## 2. Load the fixed split and prepare inputs

The manifest contains pair IDs, not expression targets. Targets are selected from matching rows in `de_train`.

In [5]:
SPLIT_PATH = PROJECT_DIR / "splits" / "all_cell_types_80_20_seed42.json"
split = load_or_create_split(
    de_train,
    SPLIT_PATH,
    validation_fraction=0.2,
    seed=42,
)

train_df, val_df = apply_split(de_train, split)

summary = pd.concat([
    train_df.groupby("cell_type").size().rename("train"),
    val_df.groupby("cell_type").size().rename("validation"),
], axis=1).fillna(0).astype(int)
summary["total"] = summary.sum(axis=1)
display(summary)
print(f"Train rows: {len(train_df)} | Validation rows: {len(val_df)}")

,train,validation,total
cell_type,,,
B cells,13,4,17
Myeloid cells,13,4,17
NK cells,116,30,146
T cells CD4+,116,30,146
T cells CD8+,113,29,142
T regulatory cells,116,30,146


Train rows: 487 | Validation rows: 127


In [6]:
feature_cols = ["cell_type", "sm_name"]
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
y_train = train_df[gene_cols].copy()
y_val = val_df[gene_cols].copy()
X_test = id_map[feature_cols].copy()
test_ids = id_map["id"].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert y_train.columns.equals(y_val.columns)

X_train: (487, 2) y_train: (487, 18211)
X_val:   (127, 2) y_val:   (127, 18211)
X_test:  (255, 2)


## 3. PCA + Ridge baseline

`n_components` is the number of PCs summarizing gene responses. `alpha` controls Ridge regularization; larger values shrink coefficients more. These are starting values, not tuned results.

PCA is fitted only on `y_train`. Ridge learns from training identifiers and training PC scores. Validation targets are used only for scoring.

In [ ]:
N_COMPONENTS = 160
RIDGE_ALPHA = 3.5

val_predictions = fit_predict(
    X_train=X_train,
    y_train=y_train,
    X_predict=X_val,
    n_components=N_COMPONENTS,
    alpha=RIDGE_ALPHA,
)

assert val_predictions.shape == y_val.shape
assert val_predictions.columns.equals(y_val.columns)
validation_mrrmse = mrrmse(y_val, val_predictions)
print(f"Validation MRRMSE: {validation_mrrmse:.6f}")
display(val_predictions.iloc[:5, :8])

Validation MRRMSE: 1.336586


,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,A4GALT,AAAS,AACS
275,3.506308,3.055178,3.086702,0.728189,7.464187,7.199276,0.713881,2.807808
8,0.520947,0.211623,1.438711,1.286974,3.313022,2.090941,-0.141160,0.414536
371,3.059587,1.501765,0.573694,0.996351,7.774345,7.188565,0.457249,1.537263
517,0.354840,0.329056,1.329253,0.927264,2.913217,2.227761,0.048540,0.249683
276,4.080108,3.074842,0.575789,0.691031,9.133957,7.601177,0.919394,2.890073


### Ridge baseline without PCA

Use the same training/validation split, one-hot inputs, Ridge alpha, and MRRMSE metric as above. Ridge predicts all gene DE scores directly, without reducing the targets to PCs.


In [51]:
RIDGE_ALPHA_RIDGE = 4.5

ridge_val_predictions = fit_predict_ridge(
    X_train=X_train,
    y_train=y_train,
    X_predict=X_val,
    alpha=RIDGE_ALPHA_RIDGE,
)

ridge_validation_mrrmse = mrrmse(y_val, ridge_val_predictions)
print(f"Ridge validation MRRMSE: {ridge_validation_mrrmse:.6f}")
display(ridge_val_predictions.iloc[:5, :8])

# Compare both models using the same split, alpha, and metric.
display(pd.DataFrame({
    "Model": ["PCA + Ridge", "Ridge"],
    "Validation MRRMSE": [validation_mrrmse, ridge_validation_mrrmse],
}))

Ridge validation MRRMSE: 1.341727


,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,A4GALT,AAAS,AACS
275,3.205620,2.776563,2.759464,0.748337,6.861724,6.587988,0.619713,2.544916
8,0.588885,0.335781,1.132126,1.303764,3.212062,2.090418,0.004962,0.428584
371,2.791353,1.378817,0.513915,0.998460,7.095236,6.509395,0.401389,1.372962
517,0.414980,0.382133,1.170110,0.827227,2.895683,2.187651,0.046732,0.315286
276,3.738203,2.741400,0.448554,0.701911,8.503576,7.037912,0.772010,2.667796


,Model,Validation MRRMSE
0,PCA + Ridge,1.336586
1,Ridge,1.341727


## 4. Official test predictions after model selection

After choosing PCA dimension and Ridge alpha using validation, refit on all labeled pairs and predict the official test pairs in `id_map` order.

In [52]:
final_X_train = de_train[feature_cols].copy()
final_y_train = de_train[gene_cols].copy()
test_predictions = fit_predict_ridge(
    X_train=final_X_train,
    y_train=final_y_train,
    X_predict=X_test,
    alpha=RIDGE_ALPHA_RIDGE,
)
assert len(test_predictions) == len(id_map)
assert test_predictions.columns.equals(y_train.columns)
print("Test prediction matrix:", test_predictions.shape)
print("Prediction order follows id_map.")

Test prediction matrix: (255, 18211)
Prediction order follows id_map.


In [53]:
# Use the sample file to get Kaggle's required column order.
sample_columns = pd.read_csv(
    DATA_DIR / "sample_submission.csv", nrows=0
).columns

# Attach each prediction row to its id in id_map.
submission = pd.concat(
    [
        id_map[["id"]].reset_index(drop=True),
        test_predictions.reset_index(drop=True),
    ],
    axis=1,
)

# Check the gene names and put columns in the required order.
assert set(submission.columns) == set(sample_columns)
submission = submission[sample_columns]
assert submission.shape == (255, 18212)  # id + 18,211 genes

output_path = PROJECT_DIR / "submission_ridge.csv"
submission.to_csv(output_path, index=False)

print("Saved to:", output_path.resolve())
print("Shape:", submission.shape)
display(submission.iloc[:3, :6])

Saved to: E:\JHU_Project\Biomedical-Data-Design\Singlecell_perturbations\submission_ridge.csv
Shape: (255, 18212)


,id,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1
0,0,0.956251,0.108056,1.092294,1.955309,4.085503
1,1,0.776698,0.164406,0.936074,1.204918,3.843630
2,2,1.006954,0.312762,0.948409,1.240970,3.939112
